In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import torch

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
train = pd.read_csv('/kaggle/input/llm-classification-finetuning/train.csv')
test = pd.read_csv('/kaggle/input/llm-classification-finetuning/test.csv')

In [ ]:
X = train
X_test = test

In [ ]:
print("train NANs")
print(X.isna().sum())
print("test NANs")
print(test.isna().sum())

In [ ]:
y = np.argmax(X[["winner_model_a", "winner_model_b", "winner_tie"]].values, axis=1)

In [ ]:
del X['model_a'], X['model_b'], X['winner_model_a'], X['winner_model_b'], X['winner_tie']

In [ ]:
def strip(x):
    if isinstance(x, str):
        return x.strip("[]'\"")
    return x

for col in ["prompt", "response_a", "response_b"]:
    X[col] = X[col].apply(strip)
    X_test[col] = X_test[col].apply(strip)

X['prompt'] = (
    X['prompt'].astype(str) + ' [SEP] ' +
    X['response_a'].astype(str) + ' [VS] ' +
    X['response_b'].astype(str)
)

X_test['prompt'] = (
    X_test['prompt'].astype(str) + ' [SEP] ' +
    X_test['response_a'].astype(str) + ' [VS] ' +
    X_test['response_b'].astype(str)
)

In [ ]:
del X['response_a'], X['response_b']
del X_test['response_a'], X_test['response_b']

In [ ]:
X_test.head()

In [ ]:
y[:100]

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
model = LogisticRegression(multi_class='multinomial', max_iter=2000)
vectorizer = TfidfVectorizer(max_features=200_000, ngram_range=(1,2))

In [ ]:
X = vectorizer.fit_transform(X['prompt'])

In [ ]:
model.fit(X,y)

In [ ]:
res = model.predict(X)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

In [ ]:
acc_res = accuracy_score(y, res)
f1_res = f1_score(y, res, average='weighted')
print(acc_res)
print(f1_res)

In [ ]:
cm = confusion_matrix(y, res)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                              display_labels=['A', 'B', 'Tie'])
disp.plot(cmap='Blues', values_format='d')
plt.show()

In [ ]:
X_test = vectorizer.transform(X_test['prompt'])

In [ ]:
final = model.predict_proba(X_test)

In [ ]:
final

In [ ]:
sample = pd.read_csv('/kaggle/input/llm-classification-finetuning/sample_submission.csv')

In [ ]:
sample['id'] = test['id']
sample['winner_model_a'] = final[:,0]
sample['winner_model_b'] = final[:,1]
sample['winner_tie'] = final[:,2]

In [ ]:
sample.to_csv("submission.csv", index=False)